In [2]:
!curl -L -o input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0  4550k      0 --:--:-- --:--:-- --:--:-- 4557k


In [ ]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [4]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [5]:
text[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [ ]:
vocab = sorted(list(set(text)))
vocab_size = len(vocab)
print("".join(vocab))



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [ ]:
# let's try a simple encoder-decoder
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for i, ch in enumerate(vocab)}
encode = lambda s: [
    stoi[c] for c in s
]  # encoder: take a string, output a list of integers
decode = lambda l: "".join(
    [itos[i] for i in l]
)  # decoder: take a list of integers, output a string

print(encode("hello"))
print(decode(encode("hello")))

[46, 43, 50, 50, 53]
hello


In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [ ]:
n = int(0.9 * len(text))  # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]
block_size = 8  # chunck

In [ ]:
x = train_data[:block_size]
y = train_data[1 : block_size + 1]
for i in range(block_size):
    context = x[: i + 1]
    target = y[i]
    print(f"when input is {context} the target is {target}")


when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


`block size` is the number of characters that are used to predict the next character. -- context window

In [ ]:
torch.manual_seed(1337)
batch_size = 4  # how many independent sequences will we process in parallel?
block_size = 8  # what is the maximum context length for predictions?


def get_batch(split):
    data = train_data if split == "train" else val_data
    random_indices = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in random_indices])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in random_indices])
    return x, y


xb, yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, : t + 1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target is {target}")


inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
when input is [24] the target is 43
when input is [24, 43] the target is 58
when input is [24, 43, 58] the target is 5
when input is [24, 43, 58, 5] the target is 57
when input is [24, 43, 58, 5, 57] the target is 1
when input is [24, 43, 58, 5, 57, 1] the target is 46
when input is [24, 43, 58, 5, 57, 1, 46] the target is 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target is 39
when input is [44] the target is 53
when input is [44, 53] the target is 56
when input is [44, 53, 56] the target is 1
when input is [44, 53, 56, 1] the target is 58
when input is [44, 53, 56, 1, 58] the target is 46


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)


class BigramLanguageModel(nn.Module):
    """For this model, there is no context, the next token is only dependent on the previous token. So the feed in sequence is a lil useless here"""

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        """
        idx: (B,T) array of indices in the current context
        """
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[
                :, -1, :
            ]  # note: using list [-1] would fail because the singleton dimension is not maintained
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=-1)
        return idx


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))


torch.Size([4, 8, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [29]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-2)

In [ ]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item(), end="\r")


In [32]:
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))


Thavalleccelesislard my wsas prff ICE tryor thedunote s;
Lo;
VINo s; he fof thewhofer; mud nbengee l


time to add the context and make the model into transformer.

The mathematical trick in self-attetion

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# we want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, : t + 1]  # (t,C)
        xbow[b, t] = torch.mean(xprev, dim=0)


In [36]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [37]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

The former two loops is inefficient, we can use matrix multiplication to do it.

In [ ]:
# v2: we can use a matrix to do the mean
mean_mat = torch.tril(torch.ones(T, T))
mean_mat = mean_mat / mean_mat.sum(dim=1, keepdim=True)

xbow2 = mean_mat @ x  # (B,T,T) @ (B,T,C) -> (B,T,C)

torch.allclose(xbow, xbow2)

True

In [ ]:
# v3: use softmax
tril = torch.tril(torch.ones(T, T))
weight = torch.zeros((T, T))
weight = weight.masked_fill(tril == 0, float("-inf"))  # masking the future tokens
weight = F.softmax(
    weight, dim=1
)  # here we use equal weights, we can change this for different weights mechanism

xbow3 = weight @ x
torch.allclose(xbow, xbow3)

True

In [ ]:
# v4: self-attention
# before we are defining th weights, but for self-attention, we want to learn the weights from the data. So we have a query and a key vec, the dot product of them is the weight.
# -> if query and key align, the weight will be high.

# let's see a single head perform self-attetion: learning one kind of weight
head_size = 16
key = nn.Linear(
    C, head_size, bias=False
)  # do the linear transformation, the W matrix is randomly initialized
query = nn.Linear(C, head_size, bias=False)  # no communication between tokens so far
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)
v = value(x)

weight = q @ k.transpose(-2, -1)  # (B,T,16) @ (B,16,T) -> (B,T,T)

# mask the future tokens: this masking is for the auto-regressive property of the language model. For models like BERT, the mask can be random spot in the sequence
weight = weight.masked_fill(tril == 0, float("-inf"))
weight = F.softmax(
    weight**head_size ** (-0.5), dim=-1
)  # we add the scaling factor to keep the variance down and avoid the softmax from becoming too peaked ~ one-hot

out = weight @ v


In [ ]:
# ->